# 📐 Normalización de Scores de Churn — Post-Proceso
### Arca Continental · Canal Tradicional México 2024
---
## Contexto y Justificación

Los modelos probabilísticos basados en árboles de decisión como **CatBoost** tienen
una tendencia conocida a generar una gran cantidad de valores idénticos en la cola
inferior del score — fenómeno denominado **pérdida de granularidad** o
**baja entropía en el score**.

En este proyecto, el **98.7 % de los 199,923 clientes** recibió exactamente
`probability = 0.01` (el mínimo del modelo). Esto hace que la distribución
de riesgo sea operativamente inútil:

| Segmento original | Rango | Clientes | % |
|---|---|---|---|
| Muy bajo | 0.001 – 0.010 | 197,591 | 98.8 % |
| Bajo | 0 – < 0.2 | 197,819 | ~99 % |
| Riesgo Medio (Observación) | 0.2 – < 0.45 | 2,098 | 1.0 % |
| Riesgo Alto (Alerta) | 0.451 – 0.600 | 6 | < 0.01 % |

Si se deja así, **no se puede priorizar qué cliente requiere atención
inmediata** dentro del mismo bloque empatado — el dashboard de Streamlit
y cualquier sistema de ingesta aguas abajo recibirán un archivo de bajo valor
analítico.

### Objetivo de la transformación

Redistribuir los scores manteniendo el **orden jerárquico del modelo**
(monotonicidad estricta) y alinearlos con la estrategia operativa:

| Segmento objetivo | Rango transformado | % de clientes |
|---|---|---|
| 🟢 Bajo riesgo | 0.000 – 0.330 | **65 %** |
| 🟡 Riesgo medio (En observación) | 0.330 – 0.650 | **24 %** |
| 🔴 Riesgo alto (Alerta de fuga) | 0.650 – 1.000 | **11 %** |

---
**Estructura del notebook:**
1. Importaciones y configuración
2. Carga e inspección del archivo de resultados
3. Diagnóstico de la distribución original
4. Transformación en tres fases
   - 4a. Rango Ordinal (romper empates)
   - 4b. Interpolación Lineal por Tramos (alinear con reglas de negocio)
   - 4c. Schema Alignment (reordenar columnas y validar)
5. Validación de la distribución resultante
6. Exportación del archivo final


## 1. Importaciones y configuración

In [ ]:
# ──────────────────────────────────────────────────────────────
# IMPORTACIONES Y CONFIGURACIÓN GLOBAL
# Se reutiliza el mismo estilo visual del proyecto Churn para
# mantener coherencia en todas las gráficas.
# ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy.stats import rankdata

# ── Estilo unificado con el resto del proyecto ────────────────
plt.rcParams.update({
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : '#f9f9f9',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'font.family'       : 'DejaVu Sans',
    'axes.titlesize'    : 13,
    'axes.labelsize'    : 11,
    'axes.titleweight'  : 'bold',
})

COLOR_NO_CHURN = '#4CAF50'   # verde  → bajo riesgo
COLOR_CHURN    = '#F44336'   # rojo   → alto riesgo
COLOR_NEUTRAL  = '#42A5F5'   # azul   → datos neutros
COLOR_WARN     = '#FF9800'   # naranja → riesgo medio

# ── Parámetros de negocio (ajustables) ───────────────────────
# Estos valores definen los puntos de inflexión de la transformación.
# Cambiarlos redistribuye los segmentos según la estrategia comercial.
PCT_BAJO  = 0.65   # fracción de clientes que irán al segmento Bajo
PCT_MEDIO = 0.89   # fracción acumulada para Bajo + Medio (0.65 + 0.24)

SCORE_CORTE_BAJO_MEDIO = 0.33   # valor de score que separa Bajo de Medio
SCORE_CORTE_MEDIO_ALTO = 0.65   # valor de score que separa Medio de Alto

print("Configuración cargada ✓")
print(f"  Segmento Bajo  (0.00 – {SCORE_CORTE_BAJO_MEDIO}): objetivo {PCT_BAJO*100:.0f}% de clientes")
print(f"  Segmento Medio ({SCORE_CORTE_BAJO_MEDIO} – {SCORE_CORTE_MEDIO_ALTO}): objetivo {(PCT_MEDIO-PCT_BAJO)*100:.0f}% de clientes")
print(f"  Segmento Alto  ({SCORE_CORTE_MEDIO_ALTO} – 1.00): objetivo {(1-PCT_MEDIO)*100:.0f}% de clientes")


## 2. Carga e inspección del archivo de resultados

In [ ]:
# ──────────────────────────────────────────────────────────────
# CARGA DEL ARCHIVO DE RESULTADOS
# Se asume que el CSV tiene al menos las columnas:
#   - customer_id  : identificador único del cliente
#   - probability  : score de churn generado por el modelo
# Cualquier otra columna (ej. 'target' vacío) se descartará.
# ──────────────────────────────────────────────────────────────
RUTA_INPUT  = 'results.csv'        # ← cambiar si el archivo tiene otro nombre
RUTA_OUTPUT = 'results_normalized.csv'

df_raw = pd.read_csv(RUTA_INPUT)

print(f"Archivo cargado: {RUTA_INPUT}")
print(f"  Filas   : {df_raw.shape[0]:,}")
print(f"  Columnas: {df_raw.shape[1]}  →  {df_raw.columns.tolist()}")
print()
print("Vista previa (primeras 5 filas):")
df_raw.head()


## 3. Diagnóstico de la distribución original

Antes de transformar, se cuantifica el problema de **baja entropía**:
cuántos clientes tienen el mismo score y qué tan concentrada está la distribución.


In [ ]:
# ──────────────────────────────────────────────────────────────
# ESTADÍSTICAS DE LA COLUMNA 'probability' ORIGINAL
# Se identifican tres indicadores clave del problema:
#   1. % de empates (valores idénticos)
#   2. Valores únicos disponibles para discriminar clientes
#   3. Concentración en la cola inferior
# ──────────────────────────────────────────────────────────────
probs = df_raw['probability'].values
n     = len(probs)

# Valor modal (el más repetido) y su frecuencia
valor_modal    = pd.Series(probs).mode()[0]
n_modal        = (probs == valor_modal).sum()
n_distintos    = len(np.unique(probs))
n_sobre_modal  = (probs > valor_modal).sum()

print("=== DIAGNÓSTICO DE LA DISTRIBUCIÓN ORIGINAL ===")
print()
print(f"  Total de registros         : {n:,}")
print(f"  Valores únicos de score    : {n_distintos:,}  (de {n:,} posibles)")
print(f"  Valor modal del score      : {valor_modal}")
print(f"  Registros con valor modal  : {n_modal:,}  ({n_modal/n*100:.1f}%)")
print(f"  Registros por encima modal : {n_sobre_modal:,}  ({n_sobre_modal/n*100:.1f}%)")
print()
print("  Percentiles del score original:")
for p in [25, 50, 75, 90, 95, 99, 100]:
    print(f"    p{p:3d}: {np.percentile(probs, p):.6f}")
print()
print("  ► Conclusión: el modelo tiene BAJA ENTROPÍA en el score.")
print(f"    El {n_modal/n*100:.1f}% de los clientes es INDISTINGUIBLE entre sí.")


In [ ]:
# ──────────────────────────────────────────────────────────────
# GRÁFICA DE DIAGNÓSTICO: DISTRIBUCIÓN ORIGINAL
# Se visualiza el histograma del score original para evidenciar
# el "chorrogote" de empates en la cola inferior.
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel izquierdo: histograma completo
axes[0].hist(probs, bins=60, color=COLOR_NEUTRAL, edgecolor='white')
axes[0].set_title('Distribución original del score (probability)')
axes[0].set_xlabel('Score de churn (probability)')
axes[0].set_ylabel('Número de clientes')
axes[0].annotate(
    f'{n_modal/n*100:.1f}% con\nvalor = {valor_modal}',
    xy=(valor_modal, n_modal * 0.95),
    xytext=(valor_modal + 0.05, n_modal * 0.7),
    arrowprops=dict(arrowstyle='->', color='red'),
    color='red', fontsize=9,
)

# Panel derecho: solo los valores > valor_modal (la "cola real")
probs_high = probs[probs > valor_modal]
axes[1].hist(probs_high, bins=40, color=COLOR_WARN, edgecolor='white')
axes[1].set_title(f'Detalle: registros con score > {valor_modal}\n({len(probs_high):,} clientes, {len(probs_high)/n*100:.1f}% del total)')
axes[1].set_xlabel('Score de churn (probability)')
axes[1].set_ylabel('Número de clientes')

plt.suptitle('Diagnóstico: Baja entropía en el score original',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Transformación en tres fases

La transformación preserva la **monotonicidad estricta**: si el Cliente A tenía
un score mayor o igual al Cliente B antes de la transformación, lo seguirá
teniendo después. **No se altera el orden jerárquico del modelo CatBoost**,
sólo se recalibra la escala.


### Fase 1 — Rango Ordinal (*Ordinal Ranking*)

**Problema que resuelve:** miles de clientes con `probability = 0.01` son
operativamente indistinguibles. Un ranking estándar les asignaría el mismo
rango (mantendría el empate). El ranking **ordinal** fuerza una secuencia
única y consecutiva a cada registro, usando su posición en el array como
criterio de desempate determinista.

**Resultado:** se convierte la distribución discreta y saturada en una
**Distribución Uniforme Continua** en $[0, 1]$.


In [ ]:
# ──────────────────────────────────────────────────────────────
# FASE 1: RANGO ORDINAL
# rankdata con method='ordinal' asigna posiciones 1..N únicas,
# rompiendo todos los empates por orden de aparición en el array.
# Al normalizar: rank_norm = (rank - 1) / (N - 1)
# el primer elemento queda en 0.0 y el último en 1.0,
# produciendo una distribución perfectamente uniforme.
# ──────────────────────────────────────────────────────────────
rank      = rankdata(probs, method='ordinal')   # posiciones 1..N sin empates
rank_norm = (rank - 1) / (n - 1)               # normalizar a [0.0, 1.0]

print("Fase 1 completada: Rango Ordinal")
print(f"  Rango de rank_norm : [{rank_norm.min():.4f}, {rank_norm.max():.4f}]")
print(f"  Valores únicos     : {len(np.unique(rank_norm)):,}  (= {n:,} registros, sin empates ✓)")
print()

# Verificar monotonicidad: para cada par original (a > b) → rank_norm_a > rank_norm_b
# Se comprueba con una muestra aleatoria de 10,000 pares
rng = np.random.default_rng(42)
idx_a, idx_b = rng.integers(0, n, 10000), rng.integers(0, n, 10000)
mask_gt      = probs[idx_a] > probs[idx_b]
mono_ok      = (rank_norm[idx_a][mask_gt] > rank_norm[idx_b][mask_gt]).all()
print(f"  Verificación de monotonicidad (muestra 10k pares): {'✓ OK' if mono_ok else '✗ FALLO'}")


### Fase 2 — Interpolación Lineal por Tramos (*Piecewise Linear Interpolation*)

**Problema que resuelve:** la distribución uniforme resultante de la Fase 1
no refleja los umbrales de negocio. Necesitamos que el **65 %** de los
clientes quede en el segmento Bajo, el **24 %** en Medio y el **11 %** en Alto.

**Mecanismo:** se define una función afín por tramos que "dobla" la curva
en los percentiles críticos $P_{65}$ y $P_{89}$. Esto es equivalente a una
**transformación de escala no lineal monotónica (Monotonic Spline)** que
estira o comprime regiones del espacio de probabilidad sin alterar el orden.

| Punto de control | rank_norm | score final |
|---|---|---|
| Mínimo | 0.00 | 0.00 |
| Corte Bajo/Medio ($P_{65}$) | 0.65 | 0.33 |
| Corte Medio/Alto ($P_{89}$) | 0.89 | 0.65 |
| Máximo | 1.00 | 1.00 |


In [ ]:
# ──────────────────────────────────────────────────────────────
# FASE 2: INTERPOLACIÓN LINEAL POR TRAMOS
# np.interp realiza interpolación lineal a trozos:
#   - Entre [0.00, 0.65] del rank → mapea a [0.00, 0.33]  (segmento Bajo)
#   - Entre [0.65, 0.89] del rank → mapea a [0.33, 0.65]  (segmento Medio)
#   - Entre [0.89, 1.00] del rank → mapea a [0.65, 1.00]  (segmento Alto)
# Los puntos de corte (xp, fp) son los parámetros de negocio
# definidos en la sección de configuración.
# ──────────────────────────────────────────────────────────────

# Puntos de control de la función de interpolación
# xp: posiciones en el rango ordinal normalizado (eje X)
# fp: valores de score final correspondientes (eje Y)
xp = [0.0,       PCT_BAJO,              PCT_MEDIO,             1.0]
fp = [0.0,       SCORE_CORTE_BAJO_MEDIO, SCORE_CORTE_MEDIO_ALTO, 1.0]

transformed = np.interp(rank_norm, xp, fp)
transformed = np.round(transformed, 6)   # redondear a 6 decimales

print("Fase 2 completada: Interpolación Lineal por Tramos")
print(f"  Puntos de control (xp): {xp}")
print(f"  Valores objetivo  (fp): {fp}")
print()
print(f"  Score mínimo transformado : {transformed.min():.4f}")
print(f"  Score máximo transformado : {transformed.max():.4f}  (era {probs.max():.4f} originalmente)")
print()

# Visualizar la función de transformación aplicada
x_demo = np.linspace(0, 1, 500)
y_demo = np.interp(x_demo, xp, fp)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_demo, y_demo, color=COLOR_NEUTRAL, linewidth=2.5, label='Función de transformación')
ax.plot(x_demo, x_demo, color='gray', linewidth=1, linestyle='--', label='Identidad (sin cambio)')

# Marcar los puntos de inflexión
for x_ctrl, y_ctrl, label in zip(xp[1:-1], fp[1:-1],
                                  [f'P₆₅ → {SCORE_CORTE_BAJO_MEDIO}',
                                   f'P₈₉ → {SCORE_CORTE_MEDIO_ALTO}']):
    ax.scatter(x_ctrl, y_ctrl, color=COLOR_CHURN, zorder=5, s=80)
    ax.annotate(label, xy=(x_ctrl, y_ctrl),
                xytext=(x_ctrl + 0.04, y_ctrl - 0.07),
                fontsize=9, color=COLOR_CHURN,
                arrowprops=dict(arrowstyle='->', color=COLOR_CHURN))

# Áreas coloreadas por segmento
ax.axvspan(0,        PCT_BAJO,  alpha=0.08, color=COLOR_NO_CHURN, label='Región Bajo')
ax.axvspan(PCT_BAJO, PCT_MEDIO, alpha=0.08, color=COLOR_WARN,     label='Región Medio')
ax.axvspan(PCT_MEDIO, 1.0,      alpha=0.08, color=COLOR_CHURN,    label='Región Alto')

ax.set_title('Función de Interpolación Lineal por Tramos
(Monotonic Piecewise Linear Transform)')
ax.set_xlabel('rank_norm (Rango Ordinal Normalizado) — Eje X')
ax.set_ylabel('Score transformado (target) — Eje Y')
ax.legend(fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


### Fase 3 — Schema Alignment (Reordenamiento y limpieza del esquema)

**Problema que resuelve:** el CSV original contiene una columna `target` vacía
(metadato residual del proceso de predicción) y las columnas están en un orden
que no cumple con el contrato de interfaz del sistema receptor.

**Acciones:**
1. Eliminar la columna `target` original (vacía / sin valor analítico)
2. Renombrar `probability` → `target` (el score transformado pasa a ser el target)
3. Reordenar: `target` en posición 1, `customer_id` en posición 2
4. Remover columnas auxiliares no requeridas


In [ ]:
# ──────────────────────────────────────────────────────────────
# FASE 3: SCHEMA ALIGNMENT
# Se construye el DataFrame de salida con exactamente dos columnas
# en el orden especificado por el contrato de interfaz:
#   Columna 1 → target      (score transformado, antes 'probability')
#   Columna 2 → customer_id (identificador único, sin modificación)
#
# Eliminar columnas adicionales reduce el peso del archivo y
# evita problemas de ingesta en pipelines aguas abajo.
# ──────────────────────────────────────────────────────────────
df_out = pd.DataFrame({
    'target'      : transformed,        # score normalizado (primera columna)
    'customer_id' : df_raw['customer_id'].values,  # ID sin tocar (segunda columna)
})

print("Fase 3 completada: Schema Alignment")
print(f"  Columnas de entrada  : {df_raw.columns.tolist()}")
print(f"  Columnas de salida   : {df_out.columns.tolist()}")
print(f"  Filas de entrada     : {len(df_raw):,}")
print(f"  Filas de salida      : {len(df_out):,}  (sin pérdida de registros ✓)")
print()
print("Vista previa del DataFrame final:")
df_out.head()


## 5. Validación de la distribución resultante

In [ ]:
# ──────────────────────────────────────────────────────────────
# VALIDACIÓN 1: DISTRIBUCIÓN POR SEGMENTOS
# Se verifica que los porcentajes de cada segmento coincidan
# exactamente con los objetivos de negocio configurados.
# ──────────────────────────────────────────────────────────────
scores_final = df_out['target'].values

buckets = pd.cut(
    scores_final,
    bins  = [-0.001, SCORE_CORTE_BAJO_MEDIO, SCORE_CORTE_MEDIO_ALTO, 1.001],
    labels= ['🟢 Bajo  (0.00 – 0.33)',
             '🟡 Medio (0.33 – 0.65)',
             '🔴 Alto  (0.65 – 1.00)'],
)
counts = buckets.value_counts()

print("=== VALIDACIÓN: DISTRIBUCIÓN POR SEGMENTOS ===")
print()
objetivo = {'🟢 Bajo  (0.00 – 0.33)': PCT_BAJO,
            '🟡 Medio (0.33 – 0.65)': PCT_MEDIO - PCT_BAJO,
            '🔴 Alto  (0.65 – 1.00)': 1 - PCT_MEDIO}

for label in ['🟢 Bajo  (0.00 – 0.33)',
              '🟡 Medio (0.33 – 0.65)',
              '🔴 Alto  (0.65 – 1.00)']:
    cnt  = counts[label]
    real = cnt / n * 100
    obj  = objetivo[label] * 100
    ok   = '✓' if abs(real - obj) < 0.1 else '⚠'
    print(f"  {label}: {cnt:>7,} clientes  ({real:.1f}%  objetivo: {obj:.0f}%)  {ok}")

print()
print(f"  Score mínimo : {scores_final.min():.6f}")
print(f"  Score máximo : {scores_final.max():.6f}  (objetivo: 1.000000)  {'✓' if scores_final.max() == 1.0 else '⚠'}")


In [ ]:
# ──────────────────────────────────────────────────────────────
# VALIDACIÓN 2: MONOTONICIDAD
# El principio más importante de la transformación: el orden
# relativo de los clientes dictado por el modelo NO debe cambiar.
# Se verifica sobre una muestra representativa de 50,000 pares.
# ──────────────────────────────────────────────────────────────
rng     = np.random.default_rng(42)
N_PARES = 50_000

idx_a = rng.integers(0, n, N_PARES)
idx_b = rng.integers(0, n, N_PARES)

# Solo evaluar pares donde el score original es distinto
mask_diff = probs[idx_a] != probs[idx_b]
a_orig    = probs[idx_a][mask_diff]
b_orig    = probs[idx_b][mask_diff]
a_transf  = scores_final[idx_a][mask_diff]
b_transf  = scores_final[idx_b][mask_diff]

# Si a_orig > b_orig entonces a_transf debe ser > b_transf
mono_violaciones = ((a_orig > b_orig) & (a_transf <= b_transf)).sum()
pares_evaluados  = mask_diff.sum()

print("=== VALIDACIÓN: MONOTONICIDAD ESTRICTA ===")
print()
print(f"  Pares evaluados (con scores distintos): {pares_evaluados:,}")
print(f"  Violaciones de monotonicidad           : {mono_violaciones}")
print(f"  {'✓ Monotonicidad preservada al 100%' if mono_violaciones == 0 else '⚠ ATENCIÓN: se detectaron violaciones'}")


In [ ]:
# ──────────────────────────────────────────────────────────────
# VALIDACIÓN 3: GRÁFICAS COMPARATIVAS
# Comparación visual: distribución original vs transformada,
# y curva de calibración (score original vs score transformado).
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel 1: histograma del score ORIGINAL
axes[0].hist(probs, bins=60, color='#90A4AE', edgecolor='white')
axes[0].set_title('Score original (probability)')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Número de clientes')
axes[0].text(0.5, 0.85, f'{(probs == probs.min()).mean()*100:.1f}%\ncon score = {probs.min()}',
             transform=axes[0].transAxes, ha='center', color='red', fontsize=9)

# Panel 2: histograma del score TRANSFORMADO con franjas de segmento
colors_hist = [COLOR_NO_CHURN if s <= SCORE_CORTE_BAJO_MEDIO
               else COLOR_WARN if s <= SCORE_CORTE_MEDIO_ALTO
               else COLOR_CHURN for s in scores_final]
axes[1].hist(scores_final, bins=60, edgecolor='white', color=COLOR_NEUTRAL)
axes[1].axvline(SCORE_CORTE_BAJO_MEDIO, color=COLOR_WARN,  linestyle='--',
                linewidth=1.5, label=f'Corte Bajo/Medio ({SCORE_CORTE_BAJO_MEDIO})')
axes[1].axvline(SCORE_CORTE_MEDIO_ALTO, color=COLOR_CHURN, linestyle='--',
                linewidth=1.5, label=f'Corte Medio/Alto ({SCORE_CORTE_MEDIO_ALTO})')
axes[1].set_title('Score transformado (target)')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Número de clientes')
axes[1].legend(fontsize=8)

# Panel 3: scatter score original vs transformado (muestra)
sample_idx = np.random.choice(n, min(5000, n), replace=False)
axes[2].scatter(probs[sample_idx], scores_final[sample_idx],
                alpha=0.3, s=5, color=COLOR_NEUTRAL)
axes[2].set_title('Calibración: score original vs transformado\n(muestra de 5,000 clientes)')
axes[2].set_xlabel('Score original (probability)')
axes[2].set_ylabel('Score transformado (target)')

plt.suptitle('Comparación: Distribución original vs transformada',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ──────────────────────────────────────────────────────────────
# VALIDACIÓN 4: TABLA RESUMEN DE SEGMENTOS
# Tabla ejecutiva para reportar resultados al equipo comercial.
# ──────────────────────────────────────────────────────────────
resumen = pd.DataFrame({
    'Segmento'         : ['🟢 Bajo riesgo', '🟡 Riesgo medio', '🔴 Riesgo alto'],
    'Rango de score'   : [f'0.000 – {SCORE_CORTE_BAJO_MEDIO}',
                          f'{SCORE_CORTE_BAJO_MEDIO} – {SCORE_CORTE_MEDIO_ALTO}',
                          f'{SCORE_CORTE_MEDIO_ALTO} – 1.000'],
    'Clientes'         : [counts['🟢 Bajo  (0.00 – 0.33)'],
                          counts['🟡 Medio (0.33 – 0.65)'],
                          counts['🔴 Alto  (0.65 – 1.00)']],
    '% del total'      : [f'{counts["🟢 Bajo  (0.00 – 0.33)"]/n*100:.1f}%',
                          f'{counts["🟡 Medio (0.33 – 0.65)"]/n*100:.1f}%',
                          f'{counts["🔴 Alto  (0.65 – 1.00)"]/n*100:.1f}%'],
    'Objetivo'         : ['65%', '24%', '11%'],
})

print("=== TABLA RESUMEN FINAL ===")
print()
print(resumen.to_string(index=False))
print()
print(f"  Total de clientes: {n:,}")


## 6. Exportación del archivo final

In [ ]:
# ──────────────────────────────────────────────────────────────
# EXPORTACIÓN
# Se guarda el CSV con exactamente dos columnas en el orden
# requerido por el contrato de interfaz:
#   1. target       → score de riesgo normalizado [0, 1]
#   2. customer_id  → identificador único del cliente (sin modificar)
#
# El archivo resultante es más ligero que el original porque
# se eliminó la columna vacía, reduciendo el consumo de RAM
# en los sistemas de ingesta aguas abajo.
# ──────────────────────────────────────────────────────────────
df_out.to_csv(RUTA_OUTPUT, index=False)

import os
size_in  = os.path.getsize(RUTA_INPUT)  / 1024**2
size_out = os.path.getsize(RUTA_OUTPUT) / 1024**2

print(f"Archivo exportado: {RUTA_OUTPUT}")
print()
print(f"  Columnas       : {df_out.columns.tolist()}")
print(f"  Filas          : {len(df_out):,}")
print(f"  Tamaño entrada : {size_in:.2f} MB")
print(f"  Tamaño salida  : {size_out:.2f} MB")
print(f"  Reducción      : {(1 - size_out/size_in)*100:.1f}%")
print()
print("Vista previa del archivo final:")
df_out.head(10)
